In [ ]:
import sys
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from code.Data_Preprocessing import DataPreprocessing_melt as mf
from code.Data_Processing import TargetVariables as tv
from code.Data_Processing import DataProcessing_run as rf
from joblib import Parallel, delayed
import os
encoder = LabelEncoder()
lower_pct = 40
upper_pct = 60

In [ ]:
MAX_HDRS = 69 # 0-69
MAX_CDI = 90  # CDI: 40-90
# Load EMB35.pkl file 
df = pd.read_pickle("/planilhas/EMB35.pkl")
print(f"Patients: {df['Patient_ID'].nunique()}")

_____

- Note: For calculating the standardized Y values using HDRS/CDI raw score data, refer to the worksheet in the data folder of this project's main DOI. [Use "Patient_ID" as a cross-reference.]

B2 - YAA (S3) (Y = Delta_Y : Worse and Better)

In [ ]:
df_model_YAA_S3_B = df.copy()
df_model_YAA_S3_B = tv.standardized_binary_evolution(df_model_YAA_S3_B, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'] = df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'].map({'Worse': 0, 'Better': 1})
df_model_YAA_S3_B = df_model_YAA_S3_B.dropna(subset=['Y_Binary_Classe_Delta_Y'])
df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'] = df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'].astype(int)

In [ ]:
meta_colsYAA = ['Patient_ID', 'Y_Binary_Classe_Delta_Y']
df_model_YAA_S3_B = mf.get_embeddings_per_segment_mean_std_2D(df_model_YAA_S3_B, meta_colsYAA)
df_model_YAA_S3_B = df_model_YAA_S3_B.dropna()
print(f"Patients: {df_model_YAA_S3_B['Patient_ID'].nunique()}")

B4 - YA (S3) (Y = Delta_HDRS : Worse and Better)

In [ ]:
df_model_YA_S3_B = df.copy()
df_model_YA_S3_B = tv.standardized_binary_evolution_HDRS(df_model_YA_S3_B, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YA_S3_B['Y_Binary_Classe_HDRS'] = df_model_YA_S3_B['Y_Binary_Classe_HDRS'].map({'Worse': 0, 'Better': 1})
df_model_YA_S3_B = df_model_YA_S3_B.dropna(subset=['Y_Binary_Classe_HDRS'])
df_model_YA_S3_B['Y_Binary_Classe_HDRS'] = df_model_YA_S3_B['Y_Binary_Classe_HDRS'].astype(int)

In [ ]:
meta_colsYA = ['Patient_ID', 'Y_Binary_Classe_HDRS']
df_model_YA_S3_B = mf.get_embeddings_per_segment_mean_std_2D(df_model_YA_S3_B, meta_colsYA)
df_model_YA_S3_B = df_model_YA_S3_B.dropna()
print(f"Patients: {df_model_YA_S3_B['Patient_ID'].nunique()}")

B6 - A (S3) (Y = Delta_CDI : Worse and Better)

In [ ]:
df_model_A_S3_B = df.copy()
df_model_A_S3_B = tv.standardized_binary_evolution_CDI(df_model_A_S3_B, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_A_S3_B['Y_Binary_Classe_CDI'] = df_model_A_S3_B['Y_Binary_Classe_CDI'].map({'Worse': 0, 'Better': 1})
df_model_A_S3_B = df_model_A_S3_B.dropna(subset=['Y_Binary_Classe_CDI'])
df_model_A_S3_B['Y_Binary_Classe_CDI'] = df_model_A_S3_B['Y_Binary_Classe_CDI'].astype(int)

In [ ]:
meta_colsA = ['Patient_ID', 'Y_Binary_Classe_CDI']
df_model_A_S3_B = mf.get_embeddings_per_segment_mean_std_2D(df_model_A_S3_B, meta_colsA)
df_model_A_S3_B = df_model_A_S3_B.dropna()
print(f"Patients: {df_model_A_S3_B['Patient_ID'].nunique()}")

___

In [ ]:
datasets = [(df_model_YAA_S3_B, 'Y_Binary_Classe_Delta_Y'),
    (df_model_YA_S3_B, 'Y_Binary_Classe_HDRS'),
    (df_model_A_S3_B, 'Y_Binary_Classe_CDI')]

In [ ]:
dataframes = [item[0] for item in datasets]
for df_index in dataframes:
    cols_with_nan = df_index.columns[df_index.isna().any()]
    print("Columns with None ou NaT:", cols_with_nan.tolist())

____

Running the experiments (LOO-CV patient-independent)

In [ ]:
summary_results = []
all_results = []
for idx, (df, target_column) in enumerate(datasets):
    print(f"\nProcessing Dataset {idx+1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()    
    results_rf = Parallel(n_jobs=32, backend='loky')(delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, RandomForestClassifier(class_weight='balanced', random_state=42), rf.class_param_grid_rf
        ) for patient in unique_patients)
    print("RF Done")
    result_lr = Parallel(n_jobs=32, backend='loky')(delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, LogisticRegression(class_weight='balanced', random_state=42), rf.class_param_grid_logreg
        ) for patient in unique_patients)
    print("LogR Done")
    results_xgb = Parallel(n_jobs=4, backend='loky')(delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, xgb.XGBClassifier(n_jobs=1, random_state=42, use_label_encoder=False, eval_metric='mlogloss'), rf.class_param_grid_xgb
        ) for patient in unique_patients)
    print("XGB Done")
    results_mlp = Parallel(n_jobs=32, backend='loky')(delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, MLPClassifier(random_state=42), rf.class_param_grid_mlp
        ) for patient in unique_patients)
    print("MLP Done")
    current_results = results_rf + result_lr + results_xgb + results_mlp
    all_results.extend(current_results)
    df_current = pd.DataFrame(current_results)
    for model_name in df_current["Model"].unique():
        df_model = df_current[df_current["Model"] == model_name]
        entry = {
            "Dataset_Index": idx + 1,
            "Target_Column": target_column,
            "Model": model_name,
            "Recall_Macro_Mean": df_model["Recall"].mean(),
            "Recall_Macro_Std":  df_model["Recall"].std()}
        if "Recall_Pos" in df_model.columns:
            entry.update({
                "Recall_Pos_Mean":     df_model["Recall_Pos"].mean(),
                "Precision_Pos_Mean":  df_model["Precision"].mean(),
                "F1_Pos_Mean":         df_model["F1_Score"].mean()})
        summary_results.append(entry)
df_results = pd.DataFrame(all_results)
df_summary = pd.DataFrame(summary_results)